# Mitogenome assembly protocol testing

-----
| Name | Date | 
| - | - |
| Keiler Collier | 29 April 2026 |

## Objective

### Mitogenome assembly protocol

In the [bad_mtDNA_analysis](bad_mtDNA_analysis.ipynb), we saw that many published mitogenomes in our dataset had suffered reference contamination due to low depth.

To correct this, we ran several tests:
1. Tested two pipelines (split vs nonsplit) with high-coverage data and a congeneric reference
2. Tested whether a congeneric reference made a difference vs a heterogeneric one in high-cov data
3. Tested whether low-cov data made a difference with heterogeneric/congeneric refseqs

-----

## Methods

We assembled mitogenomes with combinations of three tools:

| Tool name | Purpose | Used in final workflow? |
| - | - | - |
|[fastp](https://github.com/OpenGene/fastp) | Trims shortread data | Y |
|[bbsplit](https://github.com/bbushnell/BBTools) | Splits out mtDNA | N |
|[NOVOPlasty](https://github.com/ndierckx/NOVOPlasty) | Seed-based mitogenome assembly | Y |

### Data used

We used two reference sequences and two individuals:

| Refseq path | Order | Species | GenBank ID |
| - | - | - | - |
| `test/refseq/Anser_anser_OZ124297.1_mitogenome.fa` | Anseriform | *Anser anser* | [OZ124297.1](https://www.ncbi.nlm.nih.gov/nuccore/OZ124297.1) |
| `test/refseq/Taeniopygia_guttata_NC_007897.1_mitogenome.fa` | Passeriform | *Taeniopygia guttata* | [NC_007897.1](https://www.ncbi.nlm.nih.gov/nuccore/NC_007897.1) |
  
| Data file path | Order | Species | Coverage |
| - | - | - | - |
| `test/data/Anser_albifrons_UWBM_43977_CSW4521_R*.fastq.gz` | Anseriformes | *Anser albifrons* | 83.9 |
| `test/data/Anser_albifrons_UAM_35428_JJW2413_R*.fastq.gz` | Anseriformes | *Anser albifrons* | 7.6 |

### Experimental design

To answer our three questions, we ran the following 6 assemblies: 4 with CSW4521, and 2 with JJW2413.
Not all combinations of factors were run due to information gained from each assembly.

To obtain our results, we referenced the appropriate `log*.txt` file in each trial's `novoplasty_*/` directory.

| Use `bbsplit`? | Conordinal refseq? | Hi-cov? | Name |
| - | - | - | - |
| Y | Y | Y | bbsplit_hi_cov_close_ref |
| Y | N | Y | bbsplit_hi_cov_far_ref |
| N | Y | Y | hi_cov_close_ref |
| N | N | Y | hi_cov_far_ref |
| N | Y | N | lo_cov_close_ref |
| N | N | N | lo_cov_far_ref |

### Implementation

See [code documentation](#code-documentation).

-----

## Results

| Name | Length (bp; merged contigs) | Cov | Assembled reads |
| - | - | - | - |
| bbsplit_hi_cov_close_ref | $17788 = 6720 + 11068$ | $363$ | $24958$ |
| bbsplit_hi_cov_far_ref | $1778$ (1 contig) | - | $1766$ |
| hi_cov_close_ref | $17788 = 6720 + 11068$ | $366$ | $24946$ |
| hi_cov_far_ref | $17794 = 6720 + 11074$ | $367$ | $24968$ |
| lo_cov_close_ref | $591$ (1 contig) | - | 928 |
| lo_cov_far_ref | $591$ (1 contig) | - | 928 |

### Split vs. nonsplit

In high-coverage data, splitting did not provide improvements to the assembled mitogenome.
Splitting also proved highly sensitive to reference selection, and produced a massively reduced dataset (3x) when a hetero-ordinal reference was selected.

| Sample | Refseq | mtDNA split yield (R1+R2) |
| - | - | - |
| `CSW_4521` | `Anser_anser_OZ124297.1_mitogenome.fa` | $4.21\text{Mb}$ |
| `CSW_4521` | `Taeniopygia_guttata_NC_007897.1_mitogenome.fa` | $1.46 \text{Mb}$ |

### Taxonomically close vs far reference sequence

In both the high and low-coverage datasets, the selection of reference sequence does not affect the assembly outcome.
Outcome appears to be wholly dictated by depth.

-----

## Discussion

### Run `bbsplit` or not?

In general, you should *not* run `bbsplit`.
Splitting yields no benefits, and in practice, starves the assembler of raw reads if an inappropriate refseq is chosen.
This can be seen in Table 5's `bbsplit_hi_cov_close_ref` and `bbsplit_hi_cov_far_ref` runs.

### Refseq selection

As documented in the original NOVOPlasty paper, assembly is not influenced by taxonomic distance from the target reads.
This holds in both high and low-coverage data.

**Comparison (Mermaid charts; render on GitHub):** axis labels are **YY** = bbsplit + congeneric, **YN** = bbsplit + heterogeneric, **NY** = no bbsplit + congeneric, **NN** = no bbsplit + heterogeneric. For average coverage, **YN** is plotted as **0** because the log has no value (failed assembly).

```mermaid
xychart-beta
  title "Merged contig length (bp; sum of contigs)"
  x-axis ["YY", "YN", "NY", "NN"]
  y-axis "bp" 0 --> 18000
  bar [17788, 1778, 17788, 17794]
```

```mermaid
xychart-beta
  title "Average organelle coverage (NOVOPlasty log)"
  x-axis ["YY", "YN", "NY", "NN"]
  y-axis "coverage" 0 --> 400
  bar [363, 0, 366, 367]
```

```mermaid
xychart-beta
  title "Assembled reads"
  x-axis ["YY", "YN", "NY", "NN"]
  y-axis "reads" 0 --> 26000
  bar [24958, 1766, 24946, 24968]
```


-----
-----

## Code documentation

Install the dependencies, and run the following code:

```{bash}
FQ1=sample1_R1.fq.gz
FQ2=sample1_R2.fq.gz
SAMPLE_NAME=sample1
OUTPUT_PREFIX=test_prefix
REFSEQ=closest_mitogenome_to_sample.fa

./scripts/assemble_mitogenome.py --fq1 $FQ1 --fq2 $FQ2 --sample_name $SAMPLE_NAME --output $OUTPUT_PREFIX --refseq $REFSEQ -m 32 -S
```

We require a refseq to let NOVOPlasty pick out a seed, but if you skip running `bbsplit` with `-S` / `--skip-bbsplit`, the refseq doesn't have to be particularly close.
This takes longer, but uses less memory, and is less sensitive to refseq/seed selection.

-----

### Options (`scripts/assemble_mitogenome.py`)

| Short | Long | Type | Description |
| - | - | - | - |
| `-1` | `--fq1` | path (file) | Input forward (R1) FASTQ, optionally gzip-compressed. |
| `-2` | `--fq2` | path (file) | Input reverse (R2) FASTQ, optionally gzip-compressed. |
| `-s` | `--sample_name` | string | Sample identifier; used for output subdirectories and NOVOPlasty project name. |
| `-o` | `--output` | path (directory) | Root output directory; creates `trim_<sample>`, optionally `bbsplit_<sample>`, and `novoplasty_<sample>` beneath it. |
| `-r` | `--refseq` | path (file) | Mitochondrial reference FASTA used as `bbsplit` bait and as the NOVOPlasty seed (`Seed Input`). |
| `-S` | `--skip-bbsplit` | flag (boolean) | If set, skip `bbsplit`; NOVOPlasty uses the `fastp` trimmed reads. `-m` is not used in this mode. |
| `-m` | `--memory` | integer (gigabytes) | Java heap for `bbsplit.sh` as `-Xmx<m>g`. Default `8`. Ignored when `--skip-bbsplit` is set. |
| `-t` | `--threads` | integer | Worker threads for `fastp` (`-w`) and `bbsplit` (`threads=`). Default: half of `os.cpu_count()`, minimum `1`. |
| `-v` | `--verbose` | flag (boolean) | Print each command and stream `fastp` / `bbsplit` / NOVOPlasty output. Default is quiet (tools run with captured output); on failure, their stderr/stdout is printed before the traceback. |

### Dependencies

Dependencies are most easily installed with the [conda](https://anaconda.org/channels/anaconda/packages/conda/overview) environment described in `environment.yml`.
It can be created with

```{bash}
conda env create -f environment.yml
conda activate mitogenome-tools
```

### Bundled assets (`resources/`)

The repo keeps read-only pipeline inputs in **`resources/`**:

- **`resources/NOVOPlasty_config.txt`** — template copied per run into `<output>/novoplasty_<sample>/`.
- **`resources/bbsplit_ref/`** — cache of reference FASTA copies (and BBTools index files next to them). Your original `--refseq` path is unchanged; `bbsplit` uses the staged copy so indices do not clutter the directory where you keep references.

You can add `resources/bbsplit_ref/` to `.gitignore` if you do not want index caches under version control.